# Part 4: School-grouped folds

*From Mohamed, `03_folds_and_pipeline` sections 2–3 and `make_folds()` in `pisa_prep.py` (branch `preprocessing-and-EDA`).*

Every model in R3 and R4 is scored on students from schools it never saw in training. Section 2.9 showed that 16–47% of maths variance lies between schools, so a school with students on both sides of a split would make scores look better than they are.

We split the **schools** into 5 folds once, save the result, and everyone uses the same file. Each round, one fold is the test set and the other four are training.

| | |
|---|---|
| Reads | `../data/clean/pisa6_clean.parquet` |
| Writes | `../data/clean/folds.csv` (one row per school) |

The in-fold preprocessing (fill, clip, scale learned on the training folds only) stays in Mohamed's `pisa_prep.py`. It comes in with the models.

# 4.1) Make the folds
`StratifiedGroupKFold` keeps each school whole (the group part) and tries to give every fold the same mix of system × mostly-below-Level-2 students (the stratified part).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

DATA = Path("../data")
PV = [f"PV{i}MATH" for i in range(1, 11)]
BELOW = [f"BELOW_L2_PV{i}" for i in range(1, 11)]
SYSTEMS = ["ARE", "SAU", "QAT", "JOR", "MAR", "PSE"]
df = pd.read_parquet(DATA / "clean/pisa6_clean.parquet", columns=["CNT", "CNTSCHID", "SENWT", "W_FSTUWT"] + PV + BELOW)

mostly_below = df[BELOW].mean(axis=1) >= 0.5
strata = df["CNT"] + "_" + np.where(mostly_below, "below", "above")

df["fold"] = -1
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2022)
for fold, (_, test) in enumerate(splitter.split(df, strata, groups=df["CNTSCHID"])):
    df.loc[df.index[test], "fold"] = fold
df["fold"].value_counts().sort_index()

fold
0    12340
1    12330
2    12365
3    12361
4    12379
Name: count, dtype: int64

# 4.2) Check the folds

In [2]:
# 1) every student has a fold and no school is split
assert (df["fold"] >= 0).all()
assert (df.groupby("CNTSCHID")["fold"].nunique() == 1).all()

# 2) folds look alike: size, % below Level 2 and mean maths (SENWT, 10 PVs)
def pv_mean(d, cols, w="SENWT"):
    return np.mean([np.average(d[c], weights=d[w]) for c in cols])

balance = pd.DataFrame({f: {
    "schools": d["CNTSCHID"].nunique(),
    "students": len(d),
    "students_%": len(d) / len(df) * 100,
    "below_L2_%": pv_mean(d, BELOW) * 100,
    "maths": pv_mean(d, PV),
} for f, d in df.groupby("fold")}).T
balance.round(1)

,schools,students,students_%,below_L2_%,maths
0,397.0,12340.0,20.0,69.3,389.6
1,396.0,12330.0,20.0,69.6,387.0
2,394.0,12365.0,20.0,71.2,384.5
3,394.0,12361.0,20.0,69.6,388.5
4,392.0,12379.0,20.0,70.1,388.6


In [3]:
# 3) every system is spread over all five folds (schools per fold)
pd.crosstab(df.drop_duplicates("CNTSCHID")["CNT"], df.drop_duplicates("CNTSCHID")["fold"]).loc[SYSTEMS]

fold,0,1,2,3,4
CNT,,,,,
ARE,167,169,167,168,169
SAU,38,39,39,39,38
QAT,47,46,46,45,45
JOR,54,52,52,52,50
MAR,37,36,35,35,35
PSE,54,54,55,55,55


> **Note:** the folds are almost the same size and difficulty: about 20% of students each, with similar shares below Level 2 and similar average maths. Every system has schools in all five folds. If a model scores differently on one fold, that comes from the schools in it, not from a lopsided split.

# 4.3) Save

In [4]:
folds = df.groupby("CNTSCHID").agg(CNT=("CNT", "first"), fold=("fold", "first"), n_students=("fold", "size")).reset_index()
folds.to_csv(DATA / "clean/folds.csv", index=False)
print(len(folds), "schools saved")
folds.head()

1973 schools saved


,CNTSCHID,CNT,fold,n_students
0,27500001,PSE,2,10
1,27500002,PSE,4,35
2,27500003,PSE,0,35
3,27500004,PSE,3,15
4,27500005,PSE,1,33


# 4.4) Using the folds in a model notebook

```python
df = pd.read_parquet("../data/clean/pisa6_clean.parquet")
folds = pd.read_csv("../data/clean/folds.csv")
df["fold"] = df["CNTSCHID"].map(folds.set_index("CNTSCHID")["fold"])

for pv in range(1, 11):              # all 10 plausible values
    for k in range(5):               # 5 rounds, each fold is the test set once
        train, test = df[df["fold"] != k], df[df["fold"] == k]
        ...
```